# Цифровой Ординатор — демо пайплайна

Прототип **Agentic/RAG** СППР по КР АГ 2024 (РКО / Минздрав РФ, ID 62_3).

> Не является медицинской рекомендацией. Пациенты синтетические.

Ячейки используют **сохранённые** результаты (`data/result_*.json`) — ноутбук открывается без API-ключа.
Живой прогон: Streamlit / `scripts/run_case.py` / опциональная ячейка в конце.


In [1]:
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

cases = ["baseline", "comorbid", "complication"]
for cid in cases:
    p = ROOT / "data" / f"result_{cid}.json"
    assert p.exists(), p
    r = json.loads(p.read_text(encoding="utf-8"))
    print(f"=== {cid} ===")
    print("patient:", (r.get("patient") or {}).get("diagnosis"))
    print("rag hits:", len(r.get("rag_hits") or []))
    print("revised_plan:", bool(r.get("revised_plan")))
    print("audit findings:", len((r.get("audit") or {}).get("findings") or []))
    print("caught:", (r.get("audit") or {}).get("caught_example", "")[:200])
    print()


=== baseline ===
patient: Артериальная гипертензия, 2 степень, риск 3 (высокий)
rag hits: 8
revised_plan: False
audit findings: 0
caught: Критических клинических/процедурных дефектов не выявлено; назначения привязаны к валидированным цитатам RAG.

=== comorbid ===
patient: Артериальная гипертензия, 3 степень; ХБП С3б А2
rag hits: 8
revised_plan: False
audit findings: 3
caught: Нет информированного согласия на инвазивные процедуры: consent_invasive=false — процедурный дефект до инвазивных вмешательств.

=== complication ===
patient: Артериальная гипертензия, 2 степень; ХБП С3а
rag hits: 14
revised_plan: True
audit findings: 2
caught: Гиперкалиемия на фоне терапии — нужна коррекция: День 3: K+=5.7, рСКФ=34.



In [2]:
r = json.loads((ROOT / "data" / "result_comorbid.json").read_text(encoding="utf-8"))
plan = r.get("plan") or {}
print("summary:", plan.get("summary"))
print("meds:")
for m in plan.get("medications") or []:
    if isinstance(m, dict):
        print(f"  - {m.get('name')} {m.get('dose')} cit={m.get('citations')}")
print()
print("audit findings:")
for f in (r.get("audit") or {}).get("findings") or []:
    print(f"  [{f.get('severity')}/{f.get('severity')}] {f.get('title')}")


summary: Пациентка 68 лет с АГ 3 степени и ХБП С3б (рСКФ 38 мл/мин/1,73 м²). В анамнезе ангионевротический отёк на эналаприл (иАПФ), что является абсолютным противопоказанием к назначению иАПФ. Текущая терапия амлодипином 5 мг недостаточна (АД 172/94 мм рт.ст.). Требуется стартовая комбинированная терапия с использованием БРА (вместо иАПФ) и АК. Диуретик не назначен в стартовом плане, так как рСКФ >30 мл/мин/1,73 м², а отёков нет; при необходимости в будущем предпочтительны петлевые диуретики из-за рСКФ <45 мл/мин/1,73 м². Обязателен мониторинг калия и рСКФ.
meds:
  - Валсартан 80 мг 1 раз в сутки (стартовая доза, титрация по АД) cit=['[1]', '[2]', '[5]']
  - Амлодипин 5 мг 1 раз в сутки (продолжить текущую дозу, при необходимости титрация до 10 мг) cit=['[1]', '[2]', '[3]']

audit findings:
  [high/high] Нет информированного согласия на инвазивные процедуры
  [medium/medium] Нарастание калия на фоне иРААС у пациента с ХБП
  [medium/medium] Отсутствие информированного согласия на инваз

In [3]:
r = json.loads((ROOT / "data" / "result_complication.json").read_text(encoding="utf-8"))
traj = r.get("trajectory") or []
print("trajectory flags:")
for p in traj:
    print(f"  day {p.get('day')}: bp={p.get('bp')} K+={(p.get('labs') or {}).get('k_mmol_l')} flags={p.get('flags')}")
rev = r.get("revised_plan") or {}
print()
print("replan trigger:", rev.get("trigger"))
print("changes:", rev.get("changes"))
print("meds after replan:")
for m in rev.get("medications") or []:
    if isinstance(m, dict):
        print(f"  - {m.get('action')} {m.get('name')} {m.get('dose')}")


trajectory flags:
  day 0: bp=164/100 K+=4.6 flags=[]
  day 3: bp=156/96 K+=5.7 flags=['hyperkalemia', 'egfr_drop', 'recalculate_plan', 'hold_or_adjust_iraas']
  day 7: bp=138/84 K+=4.8 flags=['post_correction']
  day 14: bp=138/84 K+=4.8 flags=['post_correction']

replan trigger: Гиперкалиемия (K+ 5.7 ммоль/л) и снижение рСКФ до 34 мл/мин/1,73 м² на фоне терапии иАПФ.
changes: ['Отмена периндоприла (иАПФ) в связи с абсолютным противопоказанием (K+ ≥ 5,5 ммоль/л) [1].', 'Продолжение терапии амлодипином (АК) для контроля АД [8].', 'Усиление мониторинга калия и функции почек [3].']
meds after replan:
  - stop Периндоприл (иАПФ) 0 мг
  - continue Амлодипин (АК) 5-10 мг 1 раз/сут


In [4]:
chunks = json.loads((ROOT / "data" / "guidelines" / "chunks.json").read_text(encoding="utf-8"))
print("chunks:", len(chunks))
print("sample:")
for c in chunks[:3]:
    print(f"  {c.get('id')} | p.{c.get('page')} | {c.get('section')}")


chunks: 112
sample:
  cur_start_combo | p.20 | 3.4.1 Стартовая комбинированная терапия
  cur_five_classes | p.20 | 3.4.1 Пять классов АГП
  cur_iraas_contra | p.21 | 3.4.2 Противопоказания иАПФ/БРА


## Опционально: живой прогон (нужен `.env`)

```bash
export PYTHONPATH=src
python scripts/run_case.py --case comorbid
```


In [5]:
LIVE = False
if LIVE:
    import sys
    sys.path.insert(0, str(ROOT / "src"))
    from digital_resident.agents import run_case
    live = run_case("comorbid")
    print(live["audit"].get("caught_example"))
else:
    print("LIVE=False — используйте сохранённые result_*.json выше")


LIVE=False — используйте сохранённые result_*.json выше
